In [34]:
# DISCOUNTED CASH FLOW (DCF) VALUATION MODEL
# Author: Tushar Singla
# Description: Projects free cash flows, discounts at WACC,
#              calculates intrinsic equity value per share,
#              and runs a sensitivity analysis on key inputs.

!pip install yfinance pandas -q
import yfinance as yf
import pandas as pd
import numpy as np

print("Libraries loaded.")

Libraries loaded.


In [30]:
# TARGET COMPANY
TICKER = 'AAPL'

stock = yf.Ticker(TICKER)
info = stock.info

company_name = info.get('shortName', TICKER)
current_price = info.get('currentPrice', info.get('previousClose', 0))
shares_outstanding = info.get('sharesOutstanding', 0)
total_debt = info.get('totalDebt', 0)
cash = info.get('totalCash', 0)

# Pull recent financials
financials = stock.financials
if not financials.empty:
    latest_revenue = financials.loc['Total Revenue'].iloc[0]
else:
    latest_revenue = info.get('totalRevenue', 0)

print(f"Company: {company_name} ({TICKER})")
print(f"Current Share Price: ${current_price:,.2f}")
print(f"Shares Outstanding: {shares_outstanding:,.0f}")
print(f"Latest Annual Revenue: ${latest_revenue:,.0f}")
print(f"Total Debt: ${total_debt:,.0f}")
print(f"Cash & Equivalents: ${cash:,.0f}")

Company: Apple Inc. (AAPL)
Current Share Price: $345.97
Shares Outstanding: 14,594,180,000
Latest Annual Revenue: $416,161,000,000
Total Debt: $84,343,996,416
Cash & Equivalents: $62,399,000,576


In [35]:
# MODEL ASSUMPTIONS
PROJECTION_YEARS = 5
REVENUE_GROWTH_RATE = 0.1     # 10% annual revenue growth
FCF_MARGIN = 0.28             # 28% free cash flow margin
WACC = 0.085                  # 8.5% weighted avg cost of capital
TERMINAL_GROWTH_RATE = 0.035  # 3.5% perpetuity growth rate

print("=== DCF ASSUMPTIONS ===")
print(f"Projection Period:    {PROJECTION_YEARS} years")
print(f"Revenue Growth Rate:  {REVENUE_GROWTH_RATE:.1%}")
print(f"FCF Margin:           {FCF_MARGIN:.1%}")
print(f"WACC (Discount Rate): {WACC:.1%}")
print(f"Terminal Growth Rate: {TERMINAL_GROWTH_RATE:.1%}")

=== DCF ASSUMPTIONS ===
Projection Period:    5 years
Revenue Growth Rate:  10.0%
FCF Margin:           28.0%
WACC (Discount Rate): 8.5%
Terminal Growth Rate: 3.5%


In [36]:
# PROJECT FREE CASH FLOWS
years = list(range(1, PROJECTION_YEARS + 1))
revenues = []
fcfs = []
discounted_fcfs = []

for year in years:
    rev = latest_revenue * (1 + REVENUE_GROWTH_RATE) ** year
    fcf = rev * FCF_MARGIN
    disc_fcf = fcf / (1 + WACC) ** year
    revenues.append(rev)
    fcfs.append(fcf)
    discounted_fcfs.append(disc_fcf)

# Terminal Value (Gordon Growth Model)
terminal_value = fcfs[-1] * (1 + TERMINAL_GROWTH_RATE) / (WACC - TERMINAL_GROWTH_RATE)
pv_terminal = terminal_value / (1 + WACC) ** PROJECTION_YEARS

# Enterprise Value
sum_pv_fcfs = sum(discounted_fcfs)
enterprise_value = sum_pv_fcfs + pv_terminal

# Equity Value
equity_value = enterprise_value - total_debt + cash
intrinsic_per_share = equity_value / shares_outstanding if shares_outstanding else 0

# Build results table
projection = pd.DataFrame({
    'Year': years,
    'Projected Revenue': [f"${r:,.0f}" for r in revenues],
    'Free Cash Flow': [f"${f:,.0f}" for f in fcfs],
    'Discounted FCF': [f"${d:,.0f}" for d in discounted_fcfs],
})

print("=== PROJECTED FREE CASH FLOWS ===\n")
print(projection.to_string(index=False))
print(f"\n{'='*50}")
print(f"Sum of Discounted FCFs:   ${sum_pv_fcfs:>15,.0f}")
print(f"PV of Terminal Value:     ${pv_terminal:>15,.0f}")
print(f"Enterprise Value:         ${enterprise_value:>15,.0f}")
print(f"Less: Total Debt:         ${total_debt:>15,.0f}")
print(f"Plus: Cash:               ${cash:>15,.0f}")
print(f"Equity Value:             ${equity_value:>15,.0f}")
print(f"{'='*50}")
print(f"Intrinsic Value/Share:    ${intrinsic_per_share:>15,.2f}")
print(f"Current Market Price:     ${current_price:>15,.2f}")
print(f"{'='*50}")

upside = (intrinsic_per_share / current_price - 1) * 100 if current_price else 0
if upside > 0:
    print(f">>> UPSIDE: {upside:.1f}% — potentially UNDERVALUED")
else:
    print(f">>> DOWNSIDE: {upside:.1f}% — potentially OVERVALUED")

=== PROJECTED FREE CASH FLOWS ===

 Year Projected Revenue   Free Cash Flow   Discounted FCF
    1  $457,777,100,000 $128,177,588,000 $118,136,025,806
    2  $503,554,810,000 $140,995,346,800 $119,769,242,753
    3  $553,910,291,000 $155,094,881,480 $121,425,038,736
    4  $609,301,320,100 $170,604,369,628 $123,103,725,907
    5  $670,231,452,110 $187,664,806,591 $124,805,620,736

Sum of Discounted FCFs:   $607,239,653,938
PV of Terminal Value:     $2,583,476,349,225
Enterprise Value:         $3,190,716,003,164
Less: Total Debt:         $ 84,343,996,416
Plus: Cash:               $ 62,399,000,576
Equity Value:             $3,168,771,007,324
Intrinsic Value/Share:    $         217.13
Current Market Price:     $         345.97
>>> DOWNSIDE: -37.2% — potentially OVERVALUED


In [38]:
# SENSITIVITY ANALYSIS: Intrinsic Value vs WACC & Growth Rate
wacc_range = [0.08, 0.09, 0.10, 0.11, 0.12]
growth_range = [0.020, 0.025, 0.030, 0.035, 0.040]

sensitivity = pd.DataFrame(index=[f"{w:.1%}" for w in wacc_range],
                            columns=[f"{g:.1%}" for g in growth_range])

for w in wacc_range:
    for g in growth_range:
        proj_fcfs = [latest_revenue * (1 + REVENUE_GROWTH_RATE)**y * FCF_MARGIN
                     for y in range(1, PROJECTION_YEARS + 1)]
        disc = sum(f / (1 + w)**y for y, f in enumerate(proj_fcfs, 1))
        tv = proj_fcfs[-1] * (1 + g) / (w - g)
        pv_tv = tv / (1 + w) ** PROJECTION_YEARS
        ev = disc + pv_tv
        eq = ev - total_debt + cash
        val = eq / shares_outstanding if shares_outstanding else 0
        sensitivity.loc[f"{w:.1%}", f"{g:.1%}"] = f"${val:,.0f}"

sensitivity.index.name = "WACC ↓ / Terminal Growth →"
print("=== SENSITIVITY TABLE: Intrinsic Value per Share ===\n")
print(sensitivity.to_string())
print(f"\nCurrent market price: ${current_price:,.2f}")

=== SENSITIVITY TABLE: Intrinsic Value per Share ===

                            2.0%  2.5%  3.0%  3.5%  4.0%
WACC ↓ / Terminal Growth →                              
8.0%                        $189  $204  $221  $242  $268
9.0%                        $161  $171  $183  $197  $213
10.0%                       $140  $148  $156  $166  $177
11.0%                       $124  $129  $136  $143  $151
12.0%                       $111  $115  $120  $125  $131

Current market price: $345.97
